In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os
import certifi
import time

# Force TLS to use certifi bundle (macOS fix)
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["CURL_CA_BUNDLE"] = certifi.where()

# Resolve project root when running from Notebooks_Ours/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [2]:
!pip install numpy pandas xarray scipy tqdm pystac-client planetary-computer zarr fsspec adlfs


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [4]:
# --- Filtering function (kept identical) ---
def filterg(ds, var):
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final


In [5]:
# --- Climate variable assignment function (unchanged logic) ---
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    
    return output_df

In [6]:
# Chunked TerraClimate extraction (point-only, no full-grid cache)
# Uses nearest neighbor lookups directly on the grid for each sample.

tc_vars = [
    'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'swe',
    'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi'
]

chunk_size = 200
max_retries = 3
retry_wait_seconds = 60

train_out = os.path.join(PROJECT_ROOT, 'Datasets_Ours', 'terraclimate_features_training_allvars.csv')
val_out = os.path.join(PROJECT_ROOT, 'Datasets_Ours', 'terraclimate_features_validation_allvars.csv')

expected_cols = ['Latitude', 'Longitude', 'Sample Date'] + tc_vars


def count_rows_in_csv(path: str) -> int:
    with open(path, 'r', encoding='utf-8') as f:
        return max(sum(1 for _ in f) - 1, 0)


def select_points(ds, var: str, df: pd.DataFrame) -> np.ndarray:
    times = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')
    times_filled = times.fillna(pd.Timestamp('2011-01-01'))
    lats = df['Latitude'].values
    lons = df['Longitude'].values

    points = xr.Dataset({
        'lat': (('points',), lats),
        'lon': (('points',), lons),
        'time': (('points',), times_filled.values),
    })

    vals = ds[var].sel(
        lat=points['lat'],
        lon=points['lon'],
        time=points['time'],
        method='nearest',
    ).values.astype(float)

    mask = times.isna() | pd.isna(lats) | pd.isna(lons)
    vals[mask.values] = np.nan
    return vals


def extract_chunked(input_df: pd.DataFrame, output_path: str, label: str):
    start_idx = 0
    if os.path.exists(output_path):
        existing_header = pd.read_csv(output_path, nrows=0).columns.tolist()
        if existing_header != expected_cols:
            raise ValueError(
                f"Existing file has different columns.\n"
                f"Expected: {expected_cols}\n"
                f"Found:    {existing_header}\n"
                f"Fix: delete/rename the existing file or update expected_cols."
            )
        start_idx = count_rows_in_csv(output_path)

    print(f"🚀 Running TerraClimate extraction for {label} (chunked)...")
    print(f"Total rows: {len(input_df)}")
    print(f"Output file: {output_path}")
    print(f"Chunk size: {chunk_size}")
    print(f"Resuming from row index: {start_idx}")

    ds = load_terraclimate_dataset()

    for chunk_start in range(start_idx, len(input_df), chunk_size):
        chunk_end = min(chunk_start + chunk_size, len(input_df))
        chunk_df = input_df.iloc[chunk_start:chunk_end].copy()
        print(f"\nProcessing rows {chunk_start}..{chunk_end-1} ({len(chunk_df)} rows)")

        try:
            chunk_out = chunk_df[['Latitude', 'Longitude', 'Sample Date']].copy()

            for var in tc_vars:
                for attempt in range(1, max_retries + 1):
                    try:
                        chunk_out[var] = select_points(ds, var, chunk_df)
                        break
                    except Exception as exc:
                        msg = str(exc)
                        if "AuthenticationFailed" in msg or "ClientAuthenticationError" in msg:
                            ds = load_terraclimate_dataset()
                        if attempt == max_retries:
                            raise
                        time.sleep(retry_wait_seconds)

            write_header = (not os.path.exists(output_path)) or (count_rows_in_csv(output_path) == 0)
            chunk_out.to_csv(output_path, mode='a', header=write_header, index=False)
            time.sleep(0.5)

        except Exception as exc:
            msg = str(exc)
            print(f"\n❌ Chunk failed at rows {chunk_start}..{chunk_end-1}: {msg}")
            print("You can rerun this cell to resume from the last completed chunk.")
            break


# Load data
Water_Quality_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'water_quality_training_dataset.csv'))
Validation_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'submission_template.csv'))

# Run extraction
extract_chunked(Water_Quality_df, train_out, "training")
extract_chunked(Validation_df, val_out, "validation")

# Preview
if os.path.exists(train_out):
    display(pd.read_csv(train_out).head())
if os.path.exists(val_out):
    display(pd.read_csv(val_out).head())

🚀 Running TerraClimate extraction for training (chunked)...
Total rows: 9319
Output file: /Users/aaravsonthalia/Projects/Water-Quality-Prediction/Datasets_Ours/terraclimate_features_training_allvars.csv
Chunk size: 200
Resuming from row index: 5400

Processing rows 5400..5599 (200 rows)

Processing rows 5600..5799 (200 rows)

Processing rows 5800..5999 (200 rows)

Processing rows 6000..6199 (200 rows)

Processing rows 6200..6399 (200 rows)

Processing rows 6400..6599 (200 rows)

Processing rows 6600..6799 (200 rows)

Processing rows 6800..6999 (200 rows)

Processing rows 7000..7199 (200 rows)

Processing rows 7200..7399 (200 rows)

Processing rows 7400..7599 (200 rows)

Processing rows 7600..7799 (200 rows)

Processing rows 7800..7999 (200 rows)

Processing rows 8000..8199 (200 rows)

Processing rows 8200..8399 (200 rows)

Processing rows 8400..8599 (200 rows)

Processing rows 8600..8799 (200 rows)

Processing rows 8800..8999 (200 rows)

Processing rows 9000..9199 (200 rows)

Processin

,Latitude,Longitude,Sample Date,pet,aet,def,q,ppt,soil,swe,srad,tmax,tmin,vap,vpd,ws,pdsi
0,-28.760833,17.730278,02-01-2011,236.300003,7.900000,228.400009,0.4,8.3,0.000000,0.0,317.501465,35.759998,21.029999,1.384,2.82,2.94,-1.16
1,-26.861111,28.884722,03-01-2011,134.300003,134.300003,0.000000,8.8,175.0,35.700001,0.0,240.598236,25.410000,14.099999,1.633,0.80,1.93,1.54
2,-26.450000,28.085833,03-01-2011,141.500000,141.500000,0.000000,7.8,156.1,9.900001,0.0,245.301437,26.070000,15.250000,1.638,0.93,2.15,-0.62
3,-27.671111,27.236944,03-01-2011,152.400009,152.400009,0.000000,16.1,194.7,28.200001,0.0,250.502625,27.599998,15.960000,1.587,1.18,2.22,3.08
4,-27.356667,27.286389,03-01-2011,151.900009,151.900009,0.000000,9.0,180.4,21.300001,0.0,253.000458,27.209999,16.150000,1.617,1.12,2.27,2.69


,Latitude,Longitude,Sample Date,pet,aet,def,q,ppt,soil,swe,srad,tmax,tmin,vap,vpd,ws,pdsi
0,-32.043333,27.822778,01-09-2014,114.900002,30.400000,84.500000,1.5,31.0,7.1,0.0,212.296036,24.340000,9.66,1.049,1.08,2.74,-1.49
1,-33.329167,26.077500,16-09-2015,142.500000,37.299999,105.200005,2.0,39.0,3.0,0.0,257.603882,26.420000,12.16,1.452,0.99,3.28,-0.63
2,-32.991639,27.640028,07-05-2015,84.400002,30.600000,53.799999,1.1,22.1,24.5,0.0,136.402008,24.369999,12.70,1.371,0.90,3.39,-2.15
3,-34.096389,24.439167,07-02-2012,112.900002,82.900002,30.000000,4.3,85.9,8.3,0.0,258.402496,24.330000,16.34,1.980,0.48,4.35,2.43
4,-32.000556,28.581667,01-10-2014,131.000000,18.100000,112.900002,0.9,18.0,6.4,0.0,209.501038,25.029999,12.79,1.412,0.92,3.81,-2.45
